# Proyecto de Aula: Modelado y Análisis de la Red del Metro de Madrid
## Teoría de Grafos en Python: Modelo Estación-Línea

**Asignaturas**: Algoritmos y Programación - Matemáticas Discretas  
**Semestre**: 2026-2  

---
### Enfoque de Modelado: Modelo Estación-Línea con Transbordos Reales
En lugar de un modelo simple que ignora el costo de cambiar de andén, este proyecto implementa el **Modelo Estación-Línea**:
- **Nodos**: Cada andén de cada línea es un nodo independiente `(Estación, Línea)`, ej: `Sol [L1]`, `Sol [L2]`, `Sol [L3]` (totalizando ~291 a 303 estaciones-línea).
- **Aristas de Vía (`tipo='via'`)**: Tramos ferroviarios consecutivos dentro de la misma línea, con pesos basados en la distancia y velocidad real del tren.
- **Aristas de Transbordo (`tipo='transbordo'`)**: Pasillos peatonales entre andenes dentro del mismo intercambiador con una penalización real de ~3.5 min a pie.

## 1. Carga de Datos y Construcción del Grafo

In [ ]:
import sys
import os
import importlib
sys.path.append(os.path.abspath('../src'))

# Recargar módulos para asegurar que siempre use la última versión en disco
import graph_builder, algorithms, metrics, robustness, visualization
importlib.reload(graph_builder)
importlib.reload(algorithms)
importlib.reload(metrics)
importlib.reload(robustness)
importlib.reload(visualization)

from graph_builder import load_data, build_metro_graph, get_station_platforms, get_all_physical_stations
from algorithms import compare_routes, find_best_route
from metrics import get_all_metrics
from robustness import identify_articulation_points, simulate_node_removal
from visualization import plot_graph_matplotlib, plot_graph_pyvis

# Cargar datos y construir grafo
data = load_data('../data/metro_madrid.json')
G = build_metro_graph(data)

vias = [(u, v) for u, v, d in G.edges(data=True) if d.get('tipo') == 'via']
transbordos = [(u, v) for u, v, d in G.edges(data=True) if d.get('tipo') == 'transbordo']
estaciones_fisicas = get_all_physical_stations(G)

print("=== ESTADÍSTICAS DEL GRAFO METRO DE MADRID ===")
print(f"• Total Nodos (Estaciones-Línea): {G.number_of_nodes()}")
print(f"• Total Estaciones Físicas Únicas: {len(estaciones_fisicas)}")
print(f"• Total Tramos de Tren (Vías): {len(vias)}")
print(f"• Total Pasillos de Transbordo: {len(transbordos)}")
print(f"• Total Aristas: {G.number_of_edges()}")

## 2. Cálculo de Rutas Óptimas: Menor Tiempo vs Menos Transbordos
Comparamos la ruta óptima bajo dos criterios del proyecto:
1. **Minimizar Tiempo (Dijkstra)**: Suma los minutos en tren y los minutos de caminata en transbordo.
2. **Minimizar Transbordos**: Penaliza severamente los cambios de línea para permanecer en el mismo tren.

In [ ]:
origen = "Sol"
destino = "Nuevos Ministerios"

resultado = compare_routes(G, origen, destino)

print(f"===================================================")
print(f"RUTAS DESDE '{origen}' HASTA '{destino}'")
print(f"===================================================")

print("\n--- 1. CRITERIO: MENOR TIEMPO TOTAL ---")
rt = resultado['menor_tiempo']
print(f"Tiempo Total: {rt['tiempo_total']} min (Tren: {rt['tiempo_tren']} min | Transbordo: {rt['tiempo_transbordo']} min)")
print(f"Paradas de Tren: {rt['num_paradas_tren']} | Transbordos: {rt['num_transbordos']}")
print("Itinerario paso a paso:")
for paso, inst in enumerate(rt['instrucciones'], 1):
    print(f"  {paso}. {inst}")

print("\n--- 2. CRITERIO: MENOR NÚMERO DE TRANSBORDOS ---")
rtr = resultado['menores_transbordos']
print(f"Tiempo Total: {rtr['tiempo_total']} min (Tren: {rtr['tiempo_tren']} min | Transbordo: {rtr['tiempo_transbordo']} min)")
print(f"Paradas de Tren: {rtr['num_paradas_tren']} | Transbordos: {rtr['num_transbordos']}")
print("Itinerario paso a paso:")
for paso, inst in enumerate(rtr['instrucciones'], 1):
    print(f"  {paso}. {inst}")

## 3. Métricas de Teoría de Grafos
Calculamos las métricas fundamentales de la red tanto a nivel de andenes individuales como agrupadas por estación física:

In [ ]:
metricas = get_all_metrics(G)

print("=== MÉTRICAS GLOBALES DE LA RED ===")
print(f"• Densidad de la red: {metricas['density']:.5f}")
print(f"• Diámetro de la red: {metricas['diameter']} pasos")

print("\n=== TOP 5 ESTACIONES CON MAYOR CENTRALIDAD DE INTERMEDIACIÓN ===")
top_centralidad = sorted(metricas['station_betweenness'].items(), key=lambda x: x[1], reverse=True)[:5]
for rank, (est, valor) in enumerate(top_centralidad, 1):
    print(f"  {rank}. {est}: {valor:.4f}")

print("\n=== TOP 5 ESTACIONES CON MAYOR GRADO (MÁS INTERCONECTADAS) ===")
top_grados = sorted(metricas['station_degree'].items(), key=lambda x: x[1], reverse=True)[:5]
for rank, (est, valor) in enumerate(top_grados, 1):
    print(f"  {rank}. {est}: {valor} conexiones")

## 4. Análisis de Conectividad, Robustez y Puntos Críticos
Identificamos los **Puntos de Articulación** (estaciones cuya falla desconecta partes de la red) y simulamos el impacto del corte de un nodo clave.

In [ ]:
puntos_articulacion = identify_articulation_points(G)
print(f"Total de Nodos Críticos de Articulación: {len(puntos_articulacion)}")
print(f"Ejemplos de nodos críticos: {puntos_articulacion[:5]}")

# Simulación de fallo en un nodo clave
nodo_corte = puntos_articulacion[0] if puntos_articulacion else list(G.nodes())[0]
resultado_corte = simulate_node_removal(G, nodo_corte)

print(f"\n--- SIMULACIÓN DE FALLO EN: '{nodo_corte}' ---")
print(f"¿Permanece conectada la red?: {'Sí' if resultado_corte['is_connected'] else 'No, se ha fragmentado'}")
print(f"Número de componentes resultantes: {resultado_corte['num_components']}")

## 5. Visualización del Grafo (Matplotlib y Pyvis Interactivo)

In [ ]:
# 1. Generar archivo HTML interactivo con Pyvis (se abrirá en navegador)
plot_graph_pyvis(G, output_file="../metro_madrid_pyvis.html")

# 2. Gráfico estático con Matplotlib (diferenciando vías de transbordos punteados)
plot_graph_matplotlib(G, title="Red del Metro de Madrid")